In [1]:

!pip -q install transformers datasets peft accelerate evaluate rouge_score sentencepiece

In [4]:

import torch
import transformers
import peft


Python setup successful!
PyTorch version: 2.11.0+cu130
Transformers version: 5.20.1
PEFT version: 0.21.2
GPU available: True


In [5]:

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "facebook/bart-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("BART model loaded successfully!")

model.safetensors: reconstructing file:   0%|          |  0.00B /  558MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

BART model loaded successfully!


In [6]:

article = """
The city announced a new public transportation system.
The project will introduce electric buses and improve
service frequency. Officials hope it will reduce traffic
congestion and pollution.
"""

inputs = tokenizer(
    article,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

outputs = model.generate(
    **inputs,
    max_new_tokens=50,
    num_beams=4
)

summary = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("ORIGINAL ARTICLE:")
print(article)

print("\nAI GENERATED SUMMARY:")
print(summary)


ORIGINAL ARTICLE:

The city announced a new public transportation system.
The project will introduce electric buses and improve
service frequency. Officials hope it will reduce traffic
congestion and pollution.


AI GENERATED SUMMARY:
The city announced a new public transportation system.The project will introduce electric buses and improve bus-service frequency. Officials hope it will reduce traffic-congestion and pollution.


In [5]:

from datasets import load_dataset

dataset = load_dataset(
    "abisee/cnn_dailymail",
    "3.0.0",
    split="train[:100]"
)

print("Number of articles:", len(dataset))


README.md:   0%|          | 0.00/15.6k [00:00<?, ?B/s]

3.0.0/train-00000-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  257MB            

3.0.0/train-00000-of-00003.parquet: downloading bytes:           |  0.00B            

3.0.0/train-00001-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  257MB            

3.0.0/train-00001-of-00003.parquet: downloading bytes:           |  0.00B            

...

3.0.0/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

Dataset loaded successfully!
Number of articles: 100


In [8]:

print("ORIGINAL ARTICLE:")
print(dataset[0]["article"][:1000])

print("\n" + "=" * 50)

print("HUMAN-WRITTEN SUMMARY:")
print(dataset[0]["highlights"])


ORIGINAL ARTICLE:
LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don't plan to be one of those people who, as soon as they turn 18, suddenly buy themselves a massive sports car collection or something similar," he told an Australian interviewer earlier this month. "I don't think I'll be particularly extravagant. "The things I like buying are things that cost about 10 pounds -- books and CDs and DVDs." At 18, Radcliffe will be able to gamble in a casino, buy a drink in a pub or see the horror film "Hostel: Part II," currently six places below his number one movie on the UK box office cha

In [9]:

# Get the first article from our dataset
article = dataset[0]["article"]

# Get the human-written summary
reference_summary = dataset[0]["highlights"]

# Convert the article into tokens
inputs = tokenizer(
    article,
    return_tensors="pt",
    max_length=512,
    truncation=True
)

# Ask BART to generate a summary
model.eval()

import torch

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        num_beams=4,
        early_stopping=True
    )

# Convert the output into readable text
generated_summary = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("BART GENERATED SUMMARY:")
print(generated_summary)

print("\nHUMAN-WRITTEN SUMMARY:")
print(reference_summary)


BART GENERATED SUMMARY:
LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him.Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and

HUMAN-WRITTEN SUMMARY:
Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday .
Young actor says he has no plans to fritter his cash away .
Radcliffe's earnings from first five Potter films have been held in trust fund .


In [10]:

import evaluate

# Load ROUGE evaluation metric
rouge = evaluate.load("rouge")

# Compare AI summary with human summary
scores = rouge.compute(
    predictions=[generated_summary],
    references=[reference_summary]
)

print("ROUGE RESULTS:")
print("ROUGE-1:", round(scores["rouge1"], 4))
print("ROUGE-2:", round(scores["rouge2"], 4))
print("ROUGE-L:", round(scores["rougeL"], 4))


ROUGE RESULTS:
ROUGE-1: 0.4444
ROUGE-2: 0.3304
ROUGE-L: 0.3932


In [1]:

from datasets import load_dataset
from transformers import AutoTokenizer

# Load the CNN/DailyMail dataset
dataset = load_dataset(
    "abisee/cnn_dailymail",
    "3.0.0",
    split="train[:100]"
)

# Load the BART tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "facebook/bart-base"
)

print("Number of articles:", len(dataset))


Dataset loaded successfully!
Number of articles: 100


In [2]:

# Use 20 articles for our first training test
train_data = dataset.select(range(20))

def prepare_data(examples):
    # Convert news articles into numbers
    model_inputs = tokenizer(
        examples["article"],
        max_length=256,
        truncation=True
    )

    # Convert summaries to numbers
    labels = tokenizer(
        text_target=examples["highlights"],
        max_length=64,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

tokenized_data = train_data.map(
    prepare_data,
    batched=True,
    remove_columns=train_data.column_names
)

print("Training examples ready:", len(tokenized_data))


Training examples ready: 20


In [3]:

from transformers import AutoModelForSeq2SeqLM
from peft import LoraConfig, get_peft_model, TaskType

# Load a fresh BART-base model
model = AutoModelForSeq2SeqLM.from_pretrained(
    "facebook/bart-base"
)

# Set up LoRA
lora_config = LoraConfig(
    r=4,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

# Add LoRA to BART
lora_model = get_peft_model(model, lora_config)

# Show how many parameters can be trained
lora_model.print_trainable_parameters()


Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

trainable params: 221,184 || all params: 139,641,600 || trainable%: 0.1584


In [4]:
#Trouble Shotting with AI
!pip uninstall -y torchao

In [6]:
#Trouble Shotting with AI
import importlib.metadata

try:
    version = importlib.metadata.version("torchao")
    print("torchao is installed. Version:", version)
except importlib.metadata.PackageNotFoundError:
    print("GOOD! torchao is not installed.")


GOOD! torchao is not installed.


In [7]:

from transformers import AutoModelForSeq2SeqLM
from peft import LoraConfig, get_peft_model, TaskType

# Load BART again
model = AutoModelForSeq2SeqLM.from_pretrained(
    "facebook/bart-base"
)

# Set up LoRA rank 4
lora_config = LoraConfig(
    r=4,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

# Attach LoRA to BART
lora_model = get_peft_model(model, lora_config)

# Check to see if it worked
print("SUCCESS! LoRA is ready!")
lora_model.print_trainable_parameters()


Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

SUCCESS! LoRA is ready!
trainable params: 221,184 || all params: 139,641,600 || trainable%: 0.1584


In [6]:

print("Training examples:", len(tokenized_data))
print("LoRA model is ready:", lora_model is not None)


Training examples: 20
LoRA model is ready: True


In [8]:

from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)

# Settings for our first small training experiment
training_args = Seq2SeqTrainingArguments(
    output_dir="./p08_lora_test",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    learning_rate=2e-4,
    logging_steps=2,
    save_strategy="no",
    report_to="none",
    fp16=False
)

# Prepare batches of articles and summaries
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=lora_model
)

# Create our trainer
trainer = Seq2SeqTrainer(
    model=lora_model,
    args=training_args,
    train_dataset=tokenized_data,
    data_collator=data_collator
)

# Train BART using LoRA!
trainer.train()

print("SUCCESS! LoRA training completed!")


Step,Training Loss
2,4.863992
4,4.910676
6,5.016877
8,3.953016
10,5.363050


SUCCESS! LoRA training completed!


In [9]:

# Use articles 20–24, which were not used for training
test_data = dataset.select(range(20, 25))

print("Number of test articles:", len(test_data))
print("First test article preview:")
print(test_data[0]["article"][:300])


Number of test articles: 5
First test article preview:
LAS VEGAS, Nevada (CNN)  -- Former football star O.J. Simpson will be held without bail after his arrest on robbery and assault charges, police announced late Sunday. Police released this mug shot of O.J. Simpson after his arrest. Simpson is accused of having directed several other men in an alleged


In [12]:

import torch
import evaluate
from transformers import AutoModelForSeq2SeqLM

# Load a fresh copy of original BART
baseline_model = AutoModelForSeq2SeqLM.from_pretrained(
    "facebook/bart-base"
)

# Use GPU if available
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

baseline_model = baseline_model.to(device)
lora_model = lora_model.to(device)

baseline_model.eval()
lora_model.eval()

# Load ROUGE
rouge = evaluate.load("rouge")

# Store summaries
baseline_summaries = []
lora_summaries = []
reference_summaries = []

for i, example in enumerate(test_data):
    article = example["article"]
    reference = example["highlights"]

    # Convert article into tokens
    inputs = tokenizer(
        article,
        return_tensors="pt",
        max_length=256,
        truncation=True
    ).to(device)

    # Generate summaries from both models
    with torch.no_grad():
        baseline_output = baseline_model.generate(
            **inputs,
            max_new_tokens=64,
            num_beams=4
        )

        lora_output = lora_model.generate(
            **inputs,
            max_new_tokens=64,
            num_beams=4
        )

    # Convert outputs back into text
    baseline_summary = tokenizer.decode(
        baseline_output[0],
        skip_special_tokens=True
    )

    lora_summary = tokenizer.decode(
        lora_output[0],
        skip_special_tokens=True
    )

    baseline_summaries.append(baseline_summary)
    lora_summaries.append(lora_summary)
    reference_summaries.append(reference)

print("All summaries generated!")


Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

Finished article 1/5
Finished article 2/5
Finished article 3/5
Finished article 4/5
Finished article 5/5
All summaries generated!


In [13]:

# Calculate original BART scores
baseline_scores = rouge.compute(
    predictions=baseline_summaries,
    references=reference_summaries
)

# Calculate trained LoRA scores
lora_scores = rouge.compute(
    predictions=lora_summaries,
    references=reference_summaries
)

print("ROUGE COMPARISON")
print("-" * 45)
print(f"{'Metric':<12} {'Original BART':<15} {'BART + LoRA':<15}")

for metric in ["rouge1", "rouge2", "rougeL"]:
    print(
        f"{metric:<12} "
        f"{baseline_scores[metric]:<15.4f} "
        f"{lora_scores[metric]:<15.4f}"
    )


ROUGE COMPARISON
---------------------------------------------
Metric       Original BART   BART + LoRA    
rouge1       0.3285          0.3329         
rouge2       0.1398          0.1398         
rougeL       0.2117          0.2117         


In [14]:

# Compare all 5 generated summaries

different = 0

for i in range(len(baseline_summaries)):
    if baseline_summaries[i] != lora_summaries[i]:
        different += 1

print("Total articles tested:", len(baseline_summaries))
print("Summaries that changed after LoRA:", different)
print("Summaries that stayed identical:",
      len(baseline_summaries) - different)


Total articles tested: 5
Summaries that changed after LoRA: 1
Summaries that stayed identical: 4


In [11]:
%pip install -q evaluate rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.1 MB/s eta 0:00:00


In [15]:

# Count how many summaries changed after LoRA training

different = 0

for i in range(len(baseline_summaries)):
    if baseline_summaries[i] != lora_summaries[i]:
        different += 1

print("Total articles tested:", len(baseline_summaries))
print("Summaries changed after LoRA:", different)
print("Summaries that stayed the same:",
      len(baseline_summaries) - different)


Total articles tested: 5
Summaries changed after LoRA: 1
Summaries that stayed the same: 4


In [19]:

import os
import shutil
from google.colab import files

# Create a folder for your trained model
save_path = "/content/P08_LoRA_Rank4"

os.makedirs(save_path, exist_ok=True)

# Save your trained LoRA model
lora_model.save_pretrained(save_path)

# Save the tokenizer
tokenizer.save_pretrained(save_path)

# Put everything into one ZIP file
shutil.make_archive(
    "/content/P08_LoRA_Rank4",
    "zip",
    save_path
)

print("SUCCESS! Model saved!")

# Download the ZIP file to your laptop
files.download("/content/P08_LoRA_Rank4.zip")


SUCCESS! Model saved!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>